In [ ]:
import os
import sys
import re
import glob
import json
import traceback
import subprocess
from pathlib import Path

def log(msg):
    print(msg)
    sys.stdout.flush()
    try:
        with open("execution_log.txt", "a", encoding="utf-8") as f:
            f.write(str(msg) + chr(10))
    except Exception:
        pass

try:
    log("[IBPM CR GPU] Gravando cookies do YouTube no ambiente Kaggle...")
    import base64 as _b64, zlib as _zlib
    _cookies_b64 = "eNrteleP41jS5bPmVwywvfuyYA29aaAe6EXvnV4IOpGiKBrRitgfvyrTpma6qrJ65hvMAEplIklKN26cYDBO3Kvzv/6qF9OYJX3x16Prmn9lu+56Kf4qXJriL+u6viu7rmyKd1l3Owi06vCHv92L56enrEoOru3xB4gEYQRDIIo4xKLNs7Tpskf6AFK0UrH2WrHlIxHzq83j+WN4oMeErmVh2R69u1WNNJ7p2l5OHis2ijsN7Ho20McR2fXhdl5g+JE5JIgfxUDtrYycNzSl/5JkWTe30/gHrn32iCAphIIpEjsY7ulAEiiEQ3iMkzGOxgSC4eCH4+dFBAX/Mt2TdmySqfiduXfp/Y8sYhRI/mYRx75i8e8MfTTwt8/2PgULo2Dy4PC0zR5jh9Z4R3L5A1ta0kPhmd+P//vBMITgKIhjB0fiDuW7BARBpmzgEa02566H4x5Uq51JSzxjbVo3FHsRZiHLKzZkl3AVrHOwB/5ulizoUCcbC6Qi81pLWCyaLSOFdiOHtiNHsI5iCGuX9eotKmVi0GO7cjw2MpsVNFvn055APpSht4Zu0iP32OoNQkcOMLg2DIIE/kcIPqfKrwDi2Cmy+V4AkPnnscxlh/YORbVKd1yIcFOYSL+Wn7A45BdYrJQwSOWumnOAKmAD9nKz0tUvWPpAF2pwcBuJ50RWIWMtE8foh7Eg/wSWDunMhuCMQMTU0xldVBjDjc9YPPQLLFyFtEArWJURat2uJHI3lsmvWO7EPK965bcmEcpCv2arxozV17D8Q2YdP0CgOX80h57CuLy+xS3LvSUMzseR+aUR4QCJ8Bk9iyBnvWlS2pQ+DF7xG2I9EhOdQjdeGf5v9L2/h4Rjc9Z14vvODN/kx2drmwyBt1msp0njzG6U/0Zfpt5SlGCiCttMh038wUT9n7GL/IDd75e+38WUZQ1Pd2P2aBgObx9oYYsHiRgmNvaOfAZrbVyspRepxN63k3MUkqW/OuYIcAFuRpq154QfTrkS4vGZPDqcs0vCzF2bPH86eSQmjdfDWwSV9N0VsRuxIIRfwn5TpbHvSdu0jY8IkrfzvYdkigv21NcKoBmbRq0m84eAxPGxGydApE3nAP3s2yMm7eRy97kdfogXHr+c+BxIZRl/ZHUEjaJ6EzvKJWLOAYVTppUaK+LHW6oYAViPSeWtsgOIxNgQLEkKrGyTGXI9rrJ5RGH65wRxtOsKNbDnNQLszj/kqaM5/vPZOboFwCnAcRmYtd7zlsDnPbZCUhfT/rRk5AIMEzBqvSHU93F9VmLT3grXWwpPgOtUQRZmOGNRQ1lo1gHXgTVv1FhtLkOlPMEpgdhQDQpGnN5Q1BT7PVoIZWz9kKPqh3wb3zH2/xvfPbp5mtPi59+VLXJVfDcTGtOoUtc8g1dEuiT86j4aZ06aIj1dZHjmSIZSa91GcMUiE3Hn7DLWA0d3y+YznUTSF2UrBKvLqOJ22dU6qVhpv67n6Dc6CQT2RuaIUd0E9LwO9Cii7v6xbP2JZIHMfx3EvGBnMzEuSwLWeqonZbmtPP2ZZb5kzGzdo/lSupdj3Bj4sAvqqcB/q8zFiUHNO+jvpyaOeFeyfL+t/yxE5F8IkdVWxG7PZ4NovBGmZ7By52H9BDG9fgHx0i/OTGoGU+QVCvNGKVIm+yvEznuwj4sDRxgowmcqUPA+H/NP5PPZxW8QAf6ZfZiHCNSzUA+zmTk4tv/h2C/Dgn+mH1Zt8hoAnd4AZSl107dN+wP88xZPfoAo3mDuzzHQjxhG/knD/xDP//D29MeC/h/en/7Yjf6vaVD/YOn03Srxh6PfWifeNvWf6lS/5c2faiq/YfCf7VffYhr5V5kGYRzCIBA/6E9jGIK+F9UdQwcmBaQtZ4Zm6jBkmaYhuu75wt20YYe2FUO3Sdzb9FgQ7LY2fpkw7SamJUJuqUNsm3IeVquMYC8E8PboNUVj0hpVbw7VWrJyxw1GSQo0gZCurgTaYAJiul/0WSc50BPheIF4b8yaOt3O/Q7kToaewcftGqOUcFLPjaA7+Yyoj7McblESMxCyhs201nHQoTMhk8/eYwmRGAO3k66fdKnYg3IzLc4nr7r04JBBM9iEAuVyLIITSNLWcM3HeWwlgZYrqp6r9AG6TRpw4qnQu4ohG0ExuaZIzkCb9dLSRnY/dKUD3mm6gs1p9tW4uN/uxfHkXW457HXCJQiTMlGTXrzfWoXcLHK15gfeiymouffr7HKGmPCqiDwJWlZuCwFxO/JAfSxVV67oDG7HpGVqmogaMOAqBHfZOkVKkYSjuiItq7LHcZsfy5V2NUF4tjxjdBR4WlCc0WP581YSABaYeY/TcEWyuY8UD+ZCQIBQCCuAFbRaOxpCDVu34Ex9vLNaIpXhvs4mFZEyBNOAKhlXNqhcFOm5EvD9rTX2mH1sbdftBDvubYteLH/G4uF6KwHKOCOti2HhSiV4b4sOV/u4vTZg7EZRi0KMFk4OGVQEOaExRFFxALlx3k/LiAPKFa+Z0g6MSqUX0jNcko1intesUAHvK+WfpDCKlMEGG+cql0UjFnFyn3R/eWu5+K/Zp3lbRfmv2ap5WxX7zybDrvzeFicOEvDvNiTJr25IFrdLe/n2RgwJQRB8iMvkINLQu+eLoECUJCH0wyGJkyCKwdg3TP3iFvH0AyeQA2toJu04h08DgD5/z9bPFUUwyz6F1EKUxil+hU7pKcNo/0JgRnhefXnDq9kUcrlUT6ULXPpKd5jpIT3cqgNgcb4vqEfifENU5i33Bc/q5eAI0VFysmg5QZvQGeqqWzZQH6yAR0+7lUBWyWgiKotjwW2GwcyYrY1kunT0zbpaz1vlyXDtupsPWIUDcgbtUDA/aKF9SnBWuoiLp9HPMg8/rIJ0mZBQUz27kHXQtMZFXZv7rSkKxzla16sTEqxE8ytt/fwZdNW8Z6/XT6BReSDKsI3E8AawbGCLZ0jTs12SwYc2Fi1pU+Seguc8fxLQkcvWGIhAcVuJ4DTHoug3JKV7tFhkPc6L+KTFXD3nUnPNLwrAq3hQUZYottLnAHdIRNcCQCu1j1GaDjySC0ewUwFDTf9ACP/SWVpAUCY131vVSSGQhjOsipELsBPmk1wEDb1dNMMnAtWy3HtsbXUURwQmeHEE99GpnArDTyYo5KtoUGCVZW2cYuYTJrTL7Qo0bAr1ZrLiTBmuC6dUWkm79Dd3tkEYIRAQhn5oW/y37x8+jaV59kDnss8ns4+LMkDeKgzcN/MBjiexSUsd7jDCkS8QNto24mCgcFOA/YZPkQeH5Ve+7vjFxQ8PAQzhMHbg/AOP7AsgzFid5r1ntGOThcxeT4DhCxMt33EqeozV6N9di6ZpgUnaxZLwy3ytP5xz7u7qnZ1aUqw/T2m1hfT7EDP0TeFKml6VW12r9+Z8nBLmw/v0N7HjEAyRv3RQyPvjGNmMxz2LbHphHfWZRhpyE7YshakChtkUx7LQ5eNTcUpTzthV9Ilgnhglh+Uekr3ZvApPEhmca9fTQXCda/uK5cmDUzyfIa5DTurWHXdVIlb7KpJEe128Ux6IPU0AAAIoRMhYbDgrLODfRgz3FQRh62BARyO/q4+OI1CdW04snic12pZetczyFj5CMoGzs3rnJgZpwKkRDJTB8flD98Rb7clwNzt5JtL9ZvY8dWM7qnREBLsgRTIcGeCIrluEcBL4kG5MaJJRWBHDjoBp3614E5FNEhhHjVJKRfUe5zDuEw9pyL20MbmQSdFtifbDBg99i3T9UmU2hyP3lAXPEcDFrqLnxt0M2cf5Uq+dsvgbMV4UM9wMASrXrVzkh7RPRX8b+4YSbvMVjJDEp88EA0noowxD7nLnQ8chznRxU6nK4ZXCoAca5RO/JpjWJpNwdStBibkB4XYbmPMjwsNXs9Z0SG+MOAstgb1dGHG6kdeh2Pj86N9K9FwcM6/LeD9QyGG6TJ7Oq+nYR0PlLs14E/Xb8TjQjBK1sF0+/GlKfHRazebkAZkkBbwVXt2qCRFyKHdrvFIIR93hO90c+/vIE9nOGlMTEVn5gGaH3G1SWCh/ZkXVtrkHGrV+nmz0YG8JRl/O22bq7USV2SDPLM0WOio+b9QijBfgwq4Cqu0AXYt91hKwHLoPGD9GJ2/Tb7rB5kYGuRvsPOrBvmuWP+jUGSMg0TkKLOCWpp0auZdnIdBCzL5otR/5wmDinWcB1ZobnuSB3XfWwCQEwsRBNURJjyVdMA60cLsJ8LjatFSK/NXJcrLVtqKIzsKD8uT71VOKgD86ST+fud28GK045CbPShwEDVBc3wIxzGbKYhramauiOXIrR53moAN3RDb2+yO9cD9bnoVotX4rhFOQX/3QF8gpEPNHKjSJezyd9MDmbRhCE35isqBX7Wuf+HwFuw2lnoQeT3e59QVZtW/YYO08kvt+f7rmltP0uOszF+0qwXkwTS53mjy/RBPEv2VI/3yfUu1Grj2P2hLBm06utxqh3zj8yTJE6BiB3iNEpD1v+7PtM4Eb+JsnVKdTyIyaR1VJQ0FJ44OFz0h53RwT30cczob0gH8kiI18oxZBJIIRH2L9ZcfoOofxkk8jwNY0E2orCmUTrgScgkH6aQbpm19HErbDWldVJyjr+gBLLnIemvq2ZoK34NT1DAx6NG65Z32zGP6jA8j/hANf4YlfOiCcROFn33XwPJU/1P832PlRc6XnX4bqdfnQXHrXOQvTa3rV2G3Og21UhXHSagk1OGk3OOuhsiCo7d7z/LrpdZWrIgal4joa7of36TFt/TFlVzi03r9/c3v3JIlnexczAmkhGKOp2kF04GenN36mtmf/91MH/1RCP02/XCGxn2oc/KkBf6re0EYSFIqRIAZRz3myJk7mw8dGEoQQGKUwnPy1kYSod8Dz9fkUx/Hnh1AUxeHnv9+uEu9+dQN/K0iYRD+CDFiMUGQMPJ3+AST4dyCfQ74A+dUJPmYXTIEfFnXss9FQ+Ee4+2enKveyxOa7C6cCf0mQIGcahMyA7lwxLUVePUF2MZ1aE/O8LYK1cMAxsHhs0omgMSWIsL6X0R/m/PKR+nV6bzMlo22WUlM6hisnNT737PnZEp5bECkj+nS6YGMoJiwFGn3H9q4wiZ5w7JGAFcKklUVUpb8/PfR3D9Sv04cYQDKIXlQUtDJdlF4Q8NG1pnm5Mqt5GQ2LHqVtfLavBk3hpAV63MbTyHUmFwjzm2eHCrXld6s3SlAH0+aFwxl/j4Iffkjw/0z7e/pW3C9Z8s5JuthM5uabPPARCfpcl3y1NmmfSkNwy5PbSGGo00gu6WX3KBz9uG85FOtyVa2VduqFjdTkGbzbw10nhVEPjSI7i6w0oh/r0494gfw7vfgik1EC/TKT0wzxz2F5Ap7pUPubHaKBgcX2w04QBxHcMliIat4hk772rqhrmTEKd0+6RRVTCzoaRRxsvQE7+rVU9i+p191D9FQhVWrJZiNzQm+dhCabuYnnobTAQTQ9eqsN4w/TSi6DMFlsh0kj1WesHorhTP/Y/F/msk+0rNvK3MRGQiJO7m49nOLhLyHojaYvZNjtnFyEbtb1k6ogC9cDqxzaAZ0lu0v7GeNML73USy/10ku99FIvvdRLL/XSS730Ui+91Esv9dJLvfRSL73USy/10ku99FIvvdRLL/XSS730Ui+91Esv9dJLvfRSL73USy/10ku99FL/OXop8BA57AGUuka1TvEsWy9p1b9VWvX1YH+u2eTBfy4kXcP+GG4oViWfP+ChznRcdkd4/odMmLbk088GXuNdmnsSzpM7ZNarJF4s6dKg/zfCPX9feq83xPR3HkQfiAWG3kXue7n32Mp3nPC52Gx3A5zTFtIwy6ELp2alzZH8RuKUI5rbnJPJ7nkRj81CqhuCnBhhl2AwdJgwDSwWND329ggzbK9IZzmVRc9KkPJkmHrI7oJPn2qnZFJrkfVMOishdG9wivJ9Xpi0svNzgnMAYQ9Sqc8HclGGQk8jHXNPZzE3L7LFqpwiE5EVPyNi4Kx+PzL4fV2V9H7OLyWiXu2YSzVP9jL6BqeqitJmcJ/gqDOkmJOGPc1Fbojc7nGc7oaNXWnjydebVSzPcDUhqOS28ix3WTpcVizmnvNBIH59xCV60mMkISDKcKAljIOzQEYtwXJwsUBCvsIPGJRMpzljOaK5kEUJaHajztabUvzX22Ebqmp4buwaCq8fWO2Gz8BdW1K8dhlWqrBZGHAjoLfLacmDT8fr9xL/g86NIkAEPoisGtNBbH44+EWt9rz+jq1BhbXqtZU4j6m4O21LIy1z4urMyCJeNncNc2YC2ng8H7XsKuL70TUvjUO5CkwuDX0D0AZzpP3Z5afELUvoqzXztLrGa8a85Icv+eFLfvhvkB9+aobyB8ovzDXTve67lQfFiT/iZ0vs5F645EKV/ZCJb/KzZP1Spr5rEf1DdsqMlj8yUTOx9nRucNCtT0EPnQRQXFStZnQ5kd1Y1tOWxlYrURkeu/HoqWd9eDhDYCIXRD0+cN+pboPNY9DW7UB0r7h5QYs6MddQP1oDiFdbbJkU2/PtsxFz++WcF/LtetcoOEWWhoTNGEIfeYdUOcLPzUnHensEywdbUigNkzseXBJ2meQ9TvHiom8DyHmEKJrUxbsuJz53xGFk9zv3KKE4OUI2flRmdHBvku9D45pLuZxF7a0GjhhaNTuAdUbOcWC5SpgrHoso8HFmlugVzaqygFfXicbAJ1oDKfjU9zugA88Nds70JQOnqbzy42OCjCt3xhmAIPKAzFt2SC5U2/s9+HDhEKbfcDuwr7KTXjszZkqXGNiyis27EfHvykd22va+Dj4er9+/7S/V7Es1+9+jmv3/AUIGhA=="
    if _cookies_b64.strip():
        _cookies_raw = _zlib.decompress(_b64.b64decode(_cookies_b64)).decode("utf-8")
        with open("/kaggle/working/youtube_cookies.txt", "w", encoding="utf-8") as _cf:
            _cf.write(_cookies_raw)
        log(f"[IBPM CR GPU] Cookies YouTube salvos: {len(_cookies_raw.split(chr(10)))} linhas")
    else:
        log("[IBPM CR GPU] Cookies nao disponíveis - downloads de videos bloqueados podem falhar")

    log("[IBPM CR GPU] Instalando bibliotecas no ambiente Kaggle...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "curl", "unzip"], check=False)
    # Instalar Deno - runtime JS requerido pelo yt-dlp moderno
    log("[IBPM CR GPU] Instalando Deno (JS runtime requerido pelo yt-dlp)...")
    deno_install = subprocess.run(
        ["sh", "-c", "curl -fsSL https://deno.land/install.sh | sh"],
        capture_output=True, text=True, check=False
    )
    deno_path = "/root/.deno/bin"
    os.environ["PATH"] = deno_path + ":" + os.environ.get("PATH", "")
    log(f"[IBPM CR GPU] Deno instalado. PATH: {os.environ['PATH'][:80]}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "faster-whisper", "yt-dlp"], check=False)

    import torch
    log(f"[IBPM CR GPU] PyTorch Versao: {torch.__version__} | CUDA Disponivel: {torch.cuda.is_available()}")

    model = None
    use_faster = True

    try:
        from faster_whisper import WhisperModel
        log("[IBPM CR GPU] Inicializando Faster-Whisper Large-V3 em CUDA (float32)...")
        model = WhisperModel("large-v3", device="cuda", compute_type="float32")
        log("[IBPM CR GPU] Modelo Faster-Whisper Large-V3 GPU ativado com sucesso!")
    except Exception as e_cuda:
        log(f"[IBPM CR GPU] Falha ao carregar CUDA ({e_cuda}). Ativando CTranslate2 CPU (int8 4-threads)...")
        try:
            from faster_whisper import WhisperModel
            model = WhisperModel("large-v3", device="cpu", compute_type="int8", cpu_threads=4)
            log("[IBPM CR GPU] Modelo Faster-Whisper CPU INT8 ativado!")
        except Exception as e_cpu:
            log(f"[IBPM CR GPU] Falha no CTranslate2 CPU ({e_cpu}), carregando OpenAI Whisper...")
            import whisper
            model = whisper.load_model("large-v3", device="cpu")
            use_faster = False
            log("[IBPM CR GPU] Modelo OpenAI Whisper CPU carregado!")

    pendentes = ["169_2024-05-08_TTsj3E_ebLI_culto_das_perolas_08_05_24.webm", "177_2024-05-30_ILzAvDqYI8Y_quinta_profetica_rompendo_limites_30_05_24.webm", "049_2023-04-09_NYr_VS-wKVA_domingo_culto_de_pascoa_09_04_2023.webm", "050_2023-04-10_g4W4CMMonu0_domingo_culto_de_pascoa_09_04_2023.webm", "058_2023-05-04_p5i1LCvzWvE_quinta_profetica_da_familia_04_05_2023.webm", "060_2023-05-07_2hhL-Tq2QiY_domingo_santa_ceia_07_05_2023.webm", "085_2023-08-13_iBFl3QCw05A_culto_dia_dos_pais_13_08_23.webm", "128_2024-01-04_wMtdeSP_PBk_quinta_profetica_avivamento_e_intimidade_04_01_23.webm", "006_2022-10-24_dTQ7deZHgH4_domingo_culto_de_celebrecao_23_10_2022.webm", "089_2023-08-25_NTEJejlgBi8_quinta_feira_profetica_24_08_23.webm", "097_2023-09-28__Lg0K3V_Q_A_quinta_profetica_sala_de_adoracao_28_09_23.webm", "145_2024-02-24_RZeH_28hnQw_mini_vigilia_avivamento_e_intimidade_23_02_24.webm", "148_2024-03-04_NddisUFEUyI_domingo_santa_ceia_03_03_24.webm", "156_2024-04-01_eCrTjaH1j7I_domingo_de_celebracao_culto_de_pascoa_31_03_24.webm", "160_2024-04-12_kSxBUPt9Bvg_quinta_profetica_derrubando_as_mulharas_da_minha_vida_11_04_.webm", "164_2024-04-22_lcQfI-svRrA_domingo_de_celebracao_21_04_24.webm", "227_2024-10-25_jfE4kRU7Mrc_quinta_profetica_teu_nome_e_cura_24_10_24.webm", "228_2024-10-28__6EppQWo75w_domingo_intimidade_27_10_24.webm", "229_2024-11-01_60Mn7cT3_nE_quinta_profetica_teu_nome_e_cura_31_10_24.webm", "239_2024-12-06_MK6w25_MKXw_quinta_profetica_da_familia_05_12_24.webm", "250_2025-01-03_o-CAO5cQPDQ_quinta_profetica_02_01_24.webm", "267_2025-03-03_UR11tNNKReM_domingo_de_celebracao_02_03_25.webm", "271_2025-03-13_ebNluIDhd90_quarta_profetica_chame_a_existencia_12_03_25.webm", "274_2025-03-20_fSG96TmMUk4_quarta_profetica_chame_a_existencia_19_03_25.webm", "279_2025-04-02_vIEQEx6KisA_primeiro_dia_conferencia_01_04_25.webm", "286_2025-04-21_qAMlfgJlqKI_culto_de_pascoa_20_04_25.webm", "290_2025-05-04_R8GgaByob8g_culto_de_santa_ceia_04_05_25.webm", "291_2025-05-05_NNVPhI5GTRA_culto_de_santa_ceia_04_05_25.webm", "301_2025-06-05_EVldzg5G1Os_quarta_profetica_atos_2_04_06_25.webm", "308_2025-07-03_Wg5qbNI_X-w_quarta_profetica_faz_de_novo_02_07_25.webm", "311_2025-07-09_YQPyEe19uAw_2_dia_festividade_maa_09_07_25.webm", "321_2025-07-28__BzhyHyPI5M_domingo_culto_do_amigo_27_07_25.webm", "323_2025-07-31_RtFG7tRir9I_quarta_profetica_faz_de_novo_30_07_25.webm", "327_2025-08-14_WBn23cPHELo_quarta_profetica_alegrai_vos_13_08_25.webm", "359_2025-11-13_pIyDow9iaZs_quarta_profetica_o_desafio_da_fe_12_11_25.webm", "370_2025-12-18__fom7Gyo6K0_quarta_profetica_profundidade_17_12_2025.webm", "371_2025-12-22__nI2Wifu2Ys_domingo_culto_de_natal_21_12_2025.webm", "375_2026-01-08_1KvwI8L7Um4_quarta_profetica_efata_07_01_26.webm", "379_2026-01-19__RAW9ShOUZE_domingo_de_celebracao_18_01_26.webm", "382_2026-01-29_yq3Vcl5zl1I_quarta_profetica_efata_28_01_26.webm", "383_2026-02-16_2siKjuEmpq0_domingo_de_celebracao_15_02_26.webm", "388_2026-03-05_gFeXeSlsEuE_quarta_profetica_esforca_te_04_03_26.webm", "396_2026-03-30_ECFjGc3049g_domingo_de_celebracao_29_03_26.webm", "399_2026-04-09_o2J7qjqheSo_quarta_profetica_a_cruz_08_04_26.webm", "400_2026-04-12_H8Q3dsXdLlI_domingo_de_celebracao_12_04_26.webm", "408_2026-04-30_19N573Txx0w_quarta_profetica_a_cruz_29_04_26.webm", "410_2026-05-04_ZbiTyRI2PHQ_domingo_santa_ceia_03_05_26.webm", "423_2026-06-01_CRDG6bnhBXA_domingo_de_celebracao_31_05_26.webm", "430_2026-06-22_eUlr3RNeNsE_domingo_de_celebracao_21_06_26.webm", "432_2026-07-02_Fp3jmgbz608_quarta_profetica_restituicao_01_07_26.webm", "435_2026-07-13_Izk3my2j3uQ_domingo_de_celebracao_12_07_26.webm", "436_2026-07-16_IaqUSzEzuxo_quarta_profetica_restituicao_15_07_26.webm", "437_2026-07-18_XqLuz7HRv_M_mini_vigilia_reformando_o_altar_17_07_26.webm", "441_2026-07-27_5t26RhzBOA0_domingo_sala_de_adoracao_26_07_26.webm", "442_2026-07-27_5NwIiPBdVfQ_domingo_sala_de_adoracao_26_07_26.webm", "445_2026-08-10_nDSulaP76b8_domingo_de_celebracao_09_08_26.webm"]
    log(f"[IBPM CR GPU] Total de cultos pendentes a transcrever: {len(pendentes)}")

    out_dir = Path(".")

    def extract_video_id(filename):
        match = re.search(r'_([a-zA-Z0-9_-]{11})_', filename)
        return match.group(1) if match else None

    def format_timestamp(seconds):
        hrs = int(seconds // 3600)
        mins = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        return f"{hrs:02d}:{mins:02d}:{secs:02d}"

    for idx, audio_name in enumerate(pendentes[:100], start=1):
        vid = extract_video_id(audio_name)
        stem = Path(audio_name).stem
        txt_out = out_dir / f"{stem}.txt"
        json_out = out_dir / f"{stem}.json"

        if not vid or txt_out.exists():
            continue

        log(f"[PROGRESS {idx}/{len(pendentes)}] Baixando audio do YouTube (ID: {vid})...")
        output_template = f"/tmp/audio_{vid}.%(ext)s"
        cmd_dl = [
            "yt-dlp",
            "--extractor-args", "youtube:player_client=android,web,mweb",
            "--remote-components", "ejs:github",
            "--js-runtimes", "deno:/root/.deno/bin/deno",
            "--cookies", "/kaggle/working/youtube_cookies.txt",
            "--no-check-certificates",
            "-f", "bestaudio/best",
            "-o", output_template,
            f"https://www.youtube.com/watch?v={vid}"
        ]

        try:
            res_dl = subprocess.run(cmd_dl, capture_output=True, text=True, check=True)
        except Exception as e:
            log(f"Erro ao baixar audio {vid}: {e}")
            if hasattr(e, 'stderr') and e.stderr:
                log(f"yt-dlp stderr: {e.stderr[:300]}")
            continue

        # Encontra o arquivo de áudio baixado (.m4a, .webm, .opus)
        matches = glob.glob(f"/tmp/audio_{vid}.*")
        if not matches:
            log(f"Nenhum arquivo encontrado para {vid} em /tmp/")
            continue

        temp_audio = Path(matches[0])
        log(f"[PROGRESS {idx}/{len(pendentes)}] Transcrevendo na GPU ({temp_audio.name}): {audio_name}...")
        try:
            txt_lines = []
            seg_list = []

            if use_faster:
                segments, info = model.transcribe(str(temp_audio), language="pt", beam_size=5, vad_filter=True)
                for seg in segments:
                    ts = format_timestamp(seg.start)
                    txt_lines.append(f"[{ts}] {seg.text.strip()}")
                    seg_list.append({"start": round(seg.start, 2), "end": round(seg.end, 2), "text": seg.text.strip()})
            else:
                res = model.transcribe(str(temp_audio), language="pt")
                for seg in res.get("segments", []):
                    st = seg.get("start", 0.0)
                    txt = seg.get("text", "").strip()
                    ts = format_timestamp(st)
                    txt_lines.append(f"[{ts}] {txt}")
                    seg_list.append({"start": round(st, 2), "end": round(seg.get("end", 0.0), 2), "text": txt})

            with open(txt_out, "w", encoding="utf-8") as f:
                f.write(f"TRANSCRIÇÃO WHISPER LARGE-V3 GPU\nARQUIVO: {audio_name}\n\n" + "\n".join(txt_lines))

            with open(json_out, "w", encoding="utf-8") as f:
                json.dump({"arquivo": audio_name, "video_id": vid, "segments": seg_list}, f, ensure_ascii=False, indent=2)

            temp_audio.unlink(missing_ok=True)
            log(f"[OK {idx}/{len(pendentes)}] Transcricao concluida -> {txt_out.name}")
        except Exception as err:
            log(f"Erro ao transcrever {audio_name}: {err}")

    log("[IBPM CR GPU] PROCESSO CONCLUIDO COM SUCESSO!")
except Exception as fatal_err:
    log(f"FATAL ERROR NO KERNEL GPU: {fatal_err}")
    traceback.print_exc()
